In [ ]:
# ==============================================================================
# 0. DRIVE
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

OUT_DIR       = "/content/drive/MyDrive/qwen_pii_outputs_v3"
DATASET_5K    = "/content/drive/MyDrive/training/dataset_training_5k_v3.jsonl"

# ==============================================================================
# CONFIGURATION
# ==============================================================================
MAX_SESSION_MINUTES = 105
EPOCHS              = 1
MAX_SEQ_LENGTH      = 2048
LOAD_IN_4BIT        = True
BATCH_SIZE          = 2
GRAD_ACCUM          = 4
LEARNING_RATE       = 2e-4
WARMUP_STEPS        = 50
LORA_R              = 16
LORA_ALPHA          = 16
EVAL_EXAMPLES       = 150
PROD_PROMPT_RATIO   = 0.40
SEED                = 42

# ==============================================================================
# 1. INSTALLATION
# ==============================================================================
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], check=True)

# ==============================================================================
# 2. IMPORTS
# ==============================================================================
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from unsloth import FastModel
import torch, json, re, random, time, math
import numpy as np
from collections import defaultdict
from datasets import Dataset

# ==============================================================================
# 3. TRAINING DATASET
# ==============================================================================
if not os.path.exists(DATASET_5K):
    raise FileNotFoundError(
        f"Missing training dataset: {DATASET_5K}\n"
        "Run the Dataset-generator (DataGenerator.java) locally — it creates "
        "generated_data/dataset_training_5k.jsonl — and upload it to "
        "MyDrive/training/."
    )

n_pii, n_neg = 0, 0
with open(DATASET_5K, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        record = json.loads(line)
        if record["labels"]["entities"]:
            n_pii += 1
        else:
            n_neg += 1
print(f"Training dataset: {n_pii + n_neg} examples ({n_pii} PII / {n_neg} non-PII)")
if n_neg == 0:
    raise ValueError("The training dataset has no negative examples — are you using an old dataset? Regenerate it.")

# ==============================================================================
# 4. PROMPTS
# ==============================================================================
EXTRACT_INSTRUCTIONS = [
    "Analyze the following input and extract all PII entities:",
    "You are a cybersecurity expert. Detect all Personally Identifiable Information (PII) in the input below and return it as structured JSON:",
    "Find every piece of personal data (PII) in the following payload and list each entity:",
]

DETECTORAPP_BASE_PROMPT = (
    "You are an expert AI data validator. Your task is to analyze the provided text and detect Personally Identifiable Information (PII) or Sensitive Data.\n"
    "Respond STRICTLY with a valid JSON object. Do NOT include markdown blocks (like ```json), explanations, or any other text. Return ONLY the JSON.\n\n"
    "The JSON object MUST contain exactly these 3 fields:\n"
    "{\n"
    "  \"validationStatus\": \"Must be EXACTLY one of the predefined ENUM values listed below.\",\n"
    "  \"statusDescription\": \"A short, 1-2 sentence explanation of WHY you chose this status.\",\n"
    "  \"detectedPiiTypes\": [\"TYPE_1\", \"TYPE_2\"]\n"
    "}\n\n"
    "### VALIDATION STATUS ENUMS ###\n"
    "You MUST select exactly one of the following for the 'validationStatus' field:\n"
    "- OK: Message is safe, clean, and contains no PII or sensitive data.\n"
    "- OK_WITH_WARNING: Minor issues detected, but no PII/sensitive data leakage.\n"
    "- REJECTED: General policy violation.\n"
    "- REJECTED_AI_UNCERTAIN: You detected a possible issue but with low confidence.\n"
    "- REJECTED_SENSITIVE: High-risk data leakage (e.g., passwords, valid API keys, financial data).\n"
    "- REJECTED_PII: Personally identifiable information detected (e.g., names, addresses, phones, SSN).\n"
    "- REJECTED_UNKNOWN: Rejected due to an unknown or unclassifiable reason.\n"
    "- ERROR: Input is completely unreadable or broken.\n\n"
)

RULE_VARIANTS = [
    "",
    "You are an expert GDPR compliance assistant. Analyze the following JSON message and check "
    "if it contains any personal data according to GDPR. If yes, specify which fields contain personal data and "
    "if they comply with GDPR principles (data minimization, proper formatting, no sensitive data without consent).",
    "Then assess whether the message complies with GDPR principles, including:\n"
    "- Data minimization\n- Lawful basis for processing\n- Consent when required\n"
    "- Limited access\n- Appropriate retention\n\n"
    "If GDPR violations are found, set validationStatus to REJECTED.\n"
    "If minor GDPR concerns but no full violation, set OK_WITH_WARNING.\n"
    "If compliant, no change to validationStatus.",
    "### ALLOWED PII TYPES ###\n"
    "PERSON (includes single given names like Sarah and surname patterns like McClure), USERNAME, TITLE, "
    "ORGANIZATION, EMAIL, PHONE_NUMBER, ADDRESS, CITY, COUNTRY, COORDINATE, CREDIT_CARD, ACCOUNT_NUMBER, "
    "SWIFT_CODE, IP_ADDRESS, MAC_ADDRESS, IMEI, PASSWORD.\n"
    "### DETECTION RULES (in order of precedence) ###\n"
    "0. PERSONAL-KEY PRIORITY (overrides all exclusions): if the key contains name, first, last, given, surname, "
    "email, mail, phone, mobile, address, street, city, country, contact, owner, customer, person, dob, birth, "
    "password or secret, evaluate the value as potential PII and flag plausible names (even single given names), "
    "emails, phones, addresses and passwords.\n"
    "1. KEY-BASED REJECTION (never applies to rule 0 keys): keys containing Id, ID, _id, _gid, arn, token, key, "
    "invitation, protocol, quota, or common API parameters (filter, page, callback, expand, fields, prettyPrint, "
    "alt, view, uploadType, orderBy, maxResults) hold non-sensitive metadata and must be ignored.\n"
    "2. GIBBERISH REJECTION: random alphanumeric strings, base64, hashes and spaced gibberish are never PII.\n"
    "3. AMBIGUITY: when unsure, classify as NON-SENSITIVE, unless the value sits under a personal key.\n",
]

SENSITIVE_TYPES = {"PII_PASSWORD", "PII_CREDIT_CARD", "PII_IBAN", "PII_NATIONAL_ID"}

def create_example(record, rnd):
    raw_data = record["payload"]
    payload  = raw_data if isinstance(raw_data, str) else json.dumps(raw_data, ensure_ascii=False)
    entities = record["labels"]["entities"]
    types    = list(dict.fromkeys(e["type"] for e in entities))

    if rnd.random() < PROD_PROMPT_RATIO:
        rule = RULE_VARIANTS[rnd.randrange(len(RULE_VARIANTS))]
        user = DETECTORAPP_BASE_PROMPT + rule + "Text for analysis:\n " + payload

        if entities:
            sensitive = [t for t in types if t in SENSITIVE_TYPES]
            status    = "REJECTED_SENSITIVE" if sensitive else "REJECTED_PII"
            desc      = "Contains " + str(len(entities)) + " PII entities: " + ", ".join(types[:5]) + "."
            response  = {"validationStatus": status, "statusDescription": desc, "detectedPiiTypes": types}
        else:
            response = {"validationStatus": "OK",
                       "statusDescription": "No PII or sensitive data detected.",
                       "detectedPiiTypes": []}
    else:
        instr = EXTRACT_INSTRUCTIONS[rnd.randrange(len(EXTRACT_INSTRUCTIONS))]
        user  = instr + "\n" + payload
        response = {"entities": entities}

    return [
        {"role": "user",      "content": user},
        {"role": "assistant", "content": json.dumps(response, ensure_ascii=False)},
    ]

# ==============================================================================
# 5. MODEL AND TOKENIZER
# ==============================================================================
MODEL_CANDIDATES = [
    "unsloth/Qwen3.5-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3.5-4B",
    "Qwen/Qwen3.5-4B",
]

from huggingface_hub import snapshot_download

def download_model(repo_id, retries=3):
    last_error = None
    for i in range(1, retries + 1):
        try:
            return snapshot_download(repo_id)
        except Exception as e:
            last_error = e
            print(f"   attempt {i}/{retries} failed: {str(e)[:120]}")
            time.sleep(20)
    raise last_error

model, tokenizer = None, None
for candidate in MODEL_CANDIDATES:
    try:
        print(f"Downloading {candidate}...")
        local_dir = download_model(candidate)
        model, tokenizer = FastModel.from_pretrained(
            model_name      = local_dir,
            max_seq_length  = MAX_SEQ_LENGTH,
            load_in_4bit    = LOAD_IN_4BIT,
            full_finetuning = False,
            dtype           = torch.float16,
        )
        print(f"Loaded base: {candidate}")
        break
    except Exception as e:
        print(f"{candidate} is unavailable: {str(e)[:150]}")

if model is None:
    raise RuntimeError("No Qwen3.5-4B candidate succeeded — check the exact name on HuggingFace and your network connection.")

model = FastModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = 0,
    bias                       = "none",
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
)

tok = getattr(tokenizer, "tokenizer", tokenizer)

# ==============================================================================
# 6. DATASET PREPARATION
# ==============================================================================
rnd_build = random.Random(SEED)
texts     = []
skipped   = 0

with open(DATASET_5K, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
            record["payload"]; record["labels"]["entities"]
        except (json.JSONDecodeError, KeyError):
            skipped += 1
            continue

        messages = create_example(record, rnd_build)
        try:
            texts.append(tok.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False,
                enable_thinking=False))
        except TypeError:
            texts.append(tok.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False))

print(f"\nLoaded {len(texts)} examples (skipped {skipped} invalid)")
print("--- FORMAT CHECK (first example, last 400 chars) ---")
print(texts[0][-400:])
print("------------------------------------------------------------")

dataset = Dataset.from_dict({"text": texts})

before  = len(dataset)
dataset = dataset.filter(
    lambda ex: len(tok(ex["text"])["input_ids"]) <= MAX_SEQ_LENGTH,
    num_proc=4,
)
print(f"Retained {len(dataset)}/{before} (dropped {before - len(dataset)} too long)")

split       = dataset.train_test_split(test_size=EVAL_EXAMPLES, seed=SEED)
train_ds    = split["train"]
eval_ds     = split["test"]

EFF_BATCH   = BATCH_SIZE * GRAD_ACCUM
TOTAL_STEPS = math.ceil(len(train_ds) * EPOCHS / EFF_BATCH)

lens = [len(tok(t)["input_ids"]) for t in train_ds["text"][:500]]
print(f"Lengths (sample) -> max:{max(lens)}  p95:{int(np.percentile(lens,95))}")
print(f"Training: {len(train_ds)} examples, {EPOCHS} epochs = {TOTAL_STEPS} steps (eff. batch {EFF_BATCH})")

# ==============================================================================
# 7. STATE DETECTION
# ==============================================================================
def get_last_checkpoint(out_dir):
    if not os.path.isdir(out_dir):
        return None, 0
    checkpoints = [d for d in os.listdir(out_dir) if re.match(r"checkpoint-\d+$", d)]
    if not checkpoints:
        return None, 0
    last = max(checkpoints, key=lambda d: int(d.split("-")[1]))
    return os.path.join(out_dir, last), int(last.split("-")[1])

last_ckpt, last_step = get_last_checkpoint(OUT_DIR)
print(f"\nLast checkpoint : step {last_step}/{TOTAL_STEPS} ({100*last_step//max(TOTAL_STEPS,1)}%)")

# ==============================================================================
# 8. TRAINING
# ==============================================================================
trained_in_this_session = False

if last_step < TOTAL_STEPS:
    trained_in_this_session = True
    from trl import SFTTrainer, SFTConfig
    from transformers import TrainerCallback
    from unsloth.chat_templates import train_on_responses_only

    class SessionTimeLimit(TrainerCallback):
        def __init__(self, max_minutes):
            self.deadline = time.time() + max_minutes * 60
        def on_step_end(self, args, state, control, **kwargs):
            if time.time() >= self.deadline:
                print(f"\nSession time budget expired at step {state.global_step}. Saving checkpoint...")
                control.should_save = True
                control.should_training_stop = True
            return control

    trainer = SFTTrainer(
        model         = model,
        tokenizer     = tok,
        train_dataset = train_ds,
        eval_dataset  = eval_ds,
        args = SFTConfig(
            dataset_text_field          = "text",
            max_seq_length              = MAX_SEQ_LENGTH,
            per_device_train_batch_size = BATCH_SIZE,
            gradient_accumulation_steps = GRAD_ACCUM,
            warmup_steps                = WARMUP_STEPS,
            max_steps                   = TOTAL_STEPS,
            learning_rate               = LEARNING_RATE,
            lr_scheduler_type           = "cosine",
            optim                       = "adamw_8bit",
            fp16                        = True,
            bf16                        = False,
            logging_steps               = 10,
            save_steps                  = 15,
            save_total_limit            = 2,
            output_dir                  = OUT_DIR,
            report_to                   = "none",
            seed                        = SEED,
            packing                     = False,
            per_device_eval_batch_size  = 2,
            eval_strategy               = "no",
        ),
        callbacks = [SessionTimeLimit(MAX_SESSION_MINUTES)],
    )

    trainer = train_on_responses_only(
        trainer,
        instruction_part = "<|im_start|>user\n",
        response_part    = "<|im_start|>assistant\n",
    )

    print("\nResuming from checkpoint..." if last_ckpt else "\nStarting from scratch...")
    trainer.train(resume_from_checkpoint=last_ckpt)

    try:
        metrics = trainer.evaluate()
        print(f"\nEval loss: {metrics.get('eval_loss'):.4f}")
    except Exception as e:
        print(f"(eval skipped: {e})")

    _, last_step = get_last_checkpoint(OUT_DIR)
    print(f"\nSession completed! Step {last_step}/{TOTAL_STEPS}.")
    if last_step < TOTAL_STEPS:
        print(f"   Pause — ~{TOTAL_STEPS - last_step} steps remaining.")
else:
    print("Training is already completed — going directly to GGUF export.")

# ==============================================================================
# 9. GGUF EXPORT
# ==============================================================================
last_ckpt, last_step = get_last_checkpoint(OUT_DIR)
if last_step >= TOTAL_STEPS:
    if not trained_in_this_session:
        print("Loading trained adapter from checkpoint for export...")
        del model
        torch.cuda.empty_cache()
        model, tokenizer = FastModel.from_pretrained(
            model_name     = last_ckpt,
            max_seq_length = MAX_SEQ_LENGTH,
            load_in_4bit   = LOAD_IN_4BIT,
            dtype          = torch.float16,
        )
        tok = getattr(tokenizer, "tokenizer", tokenizer)

    print("\nConverting to GGUF (q4_k_m)...")
    try:
        model.save_pretrained_gguf(
            os.path.join(OUT_DIR, "my_qwen_pii_model_v3"),
            tok,
            quantization_method = "q4_k_m",
        )
        print("Done! GGUF is on Drive in 'qwen_pii_outputs_v3/my_qwen_pii_model_v3'.")
    except Exception as e:
        print(f"GGUF export failed: {e}")
        print("   Fallback: saving merged 16-bit model for manual conversion...")
        merged_dir = os.path.join(OUT_DIR, "my_qwen_pii_model_v3_merged16")
        model.save_pretrained_merged(merged_dir, tok, save_method="merged_16bit")
        print(f"""Merged model: {merged_dir}
Manual conversion (llama.cpp supports qwen3.5 — Ollama already serves it as GGUF):
  git clone [https://github.com/ggml-org/llama.cpp](https://github.com/ggml-org/llama.cpp) && pip install -r llama.cpp/requirements.txt
  python llama.cpp/convert_hf_to_gguf.py {merged_dir} --outfile model_v3.f16.gguf
  llama.cpp/build/bin/llama-quantize model_v3.f16.gguf model_v3.Q4_K_M.gguf q4_k_m""")

    print("""
RECOMMENDED OLLAMA MODELFILE:
FROM ./<generated-name>.Q4_K_M.gguf
PARAMETER temperature 0
PARAMETER num_ctx 4096
PARAMETER stop <|im_end|>
PARAMETER stop <|im_start|>
""")